# Локальный запуск S669 ddG

Этот notebook предназначен для запуска на текущей локальной машине. Он сначала проверяет, какие методы реально доступны, затем запускает небольшой S669 sample через локальный harness. Полный запуск по всем S669 cases лучше делать после smoke-проверки и с явным ограничением методов.

In [5]:
import sys
!{sys.executable} -m pip install biotite


   ---------------------------------------- 0.0/41.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/41.2 MB ? eta -:--:--
   - -------------------------------------- 1.0/41.2 MB 5.0 MB/s eta 0:00:09
   - -------------------------------------- 1.0/41.2 MB 5.0 MB/s eta 0:00:09
   -- ------------------------------------- 2.1/41.2 MB 2.7 MB/s eta 0:00:15
   --- ------------------------------------ 3.1/41.2 MB 3.3 MB/s eta 0:00:12
   ---- ----------------------------------- 4.7/41.2 MB 4.1 MB/s eta 0:00:09
   ----- ---------------------------------- 5.8/41.2 MB 4.2 MB/s eta 0:00:09
   ------ --------------------------------- 6.8/41.2 MB 4.2 MB/s eta 0:00:09
   ------- -------------------------------- 7.6/41.2 MB 4.2 MB/s eta 0:00:08
   -------- ------------------------------- 8.7/41.2 MB 4.3 MB/s eta 0:00:08
   --------- ------------------------------ 9.7/41.2 MB 4.3 MB/s eta 0:00:08
   ---------- ----------------------------- 10.7/41.2 MB 4.4 MB/s eta 0:00:07
   ---------

In [6]:
# 0. CONFIG
from pathlib import Path
import sys

import pandas as pd

repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

output_dir = repo_root / 'analysis' / 's669_local_run'
sample_n = 3
include_rosetta_score = False  # Rosetta score_jd2 is a structure score proxy, not a true mutation ddG protocol.
force_rerun = False

print(repo_root)
print(output_dir)


D:\Proga\OpenFold_codex\OpenFold3_project
D:\Proga\OpenFold_codex\OpenFold3_project\analysis\s669_local_run


## 1. Проверка локальной готовности

`foldx` сейчас является основным локально готовым ddG-методом. `rosetta_score` можно включить как diagnostic proxy, но его нельзя интерпретировать как полноценный ddG без отдельного Rosetta ddG-протокола.

In [7]:
# 1. PREFLIGHT
from pipeline.s669_local_run import local_capabilities, export_local_tool_env

capabilities = local_capabilities(repo_root)
display(capabilities)

exported_env = export_local_tool_env(repo_root)
for key, value in exported_env.items():
    print(f'{key}={value}')


OSError: [WinError 1920] Доступ к этому файлу из системы отсутствует: 'D:\\Proga\\OpenFold_codex\\OpenFold3_project\\tools\\bin\\foldx'

## 2. Загрузка S669

Используется уже подготовленный S669 importer. Если processed CSV отсутствует, он будет создан из архива S669.

In [3]:
# 2. LOAD S669
from pipeline.external_benchmarks import S669Config, check_s669_exists, preprocess_s669, load_s669

s669_config = S669Config()
if not check_s669_exists(s669_config):
    preprocess_s669(s669_config)

s669 = load_s669(s669_config)
print(f'S669 rows: {len(s669)}')
display(s669.groupby('direction').size().rename('rows'))
display(s669.head())


S669 rows: 1244


direction
direct     669
inverse    575
Name: rows, dtype: int64

,benchmark,direction,protein_id,pdb_id,chain,position,pdb_residue_id,wt_residue,mut_residue,experimental_ddg,mutation_id,structure_id,pdb_path,doi,temperature,ph,experimental_method,resolution
0,S669,direct,1a0fA_direct,1A0F,A,11,11,S,A,-1.80,S11A,1a0f,/mnt/d/Proga/OpenFold_codex/OpenFold3_project/...,10.1042/BJ20061707,329.83,6.5,X-RAY DIFFRACTION,2.1
1,S669,inverse,1a0fA_inverse,1A0F,A,11,11,A,S,1.80,A11S,1a0fA_S11A,/mnt/d/Proga/OpenFold_codex/OpenFold3_project/...,10.1042/BJ20061707,329.83,6.5,X-RAY DIFFRACTION,2.1
2,S669,direct,1a7vA_direct,1A7V,A,3,3,D,H,-1.36,D3H,1a7v,/mnt/d/Proga/OpenFold_codex/OpenFold3_project/...,10.1016/j.jmb.2009.07.074,298.15,6.5,X-RAY DIFFRACTION,2.3
3,S669,direct,1a7vA_direct,1A7V,A,13,13,K,H,-0.60,K13H,1a7v,/mnt/d/Proga/OpenFold_codex/OpenFold3_project/...,10.1016/j.jmb.2009.07.074,298.15,6.5,X-RAY DIFFRACTION,2.3
4,S669,direct,1a7vA_direct,1A7V,A,20,20,K,H,-2.88,K20H,1a7v,/mnt/d/Proga/OpenFold_codex/OpenFold3_project/...,10.1016/j.jmb.2009.07.074,298.15,6.5,X-RAY DIFFRACTION,2.3


## 3. Быстрый опубликованный baseline

Перед локальным пересчетом удобно посмотреть, как на этом же S669 ведут себя опубликованные prediction columns из исходного benchmark.

In [4]:
# 3. BUNDLED S669 BASELINE
from pipeline.s669_analysis import analyze_s669_published_predictions

published_outputs = analyze_s669_published_predictions()
published_metrics = pd.read_csv(published_outputs['metrics'])
display(published_metrics.head(15))

foldx_published = published_metrics[published_metrics['method'].eq('FoldX')]
display(foldx_published)


,method,representation,n,pearson,spearman,rmse,mae,bias,sign_accuracy,mean_pred,mean_exp
0,INPS-Seq,sequence,1244,0.591708,0.622288,1.550010,1.113891,0.001397,0.725080,-0.095725,-0.097122
1,PremPS,structure,1234,0.598238,0.613126,1.525927,1.091541,0.103702,0.730146,0.008955,-0.094747
2,ACDC-NN,structure,1244,0.596930,0.608973,1.524204,1.070007,0.006532,0.714630,-0.090590,-0.097122
3,ACDC-NN-Seq,sequence,1244,0.574363,0.590842,1.557982,1.100991,0.015342,0.703376,-0.081780,-0.097122
4,DDGun,structure,1190,0.553663,0.572761,1.782327,1.287042,-0.062672,0.700840,-0.164202,-0.101529
5,INPS3D,structure,1242,0.528918,0.552091,1.660107,1.200284,-0.350242,0.681965,-0.446152,-0.095910
6,DDGun3D,structure,1242,0.549391,0.533405,1.637185,1.147551,-0.031546,0.681965,-0.127456,-0.095910
7,Dynamut,structure,1244,0.485418,0.460773,1.660964,1.219330,-0.008953,0.648714,-0.106076,-0.097122
8,ThermoNet,structure,1244,0.492013,0.452983,1.656144,1.216139,0.010744,0.636656,-0.086379,-0.097122
9,PopMusic,structure,1241,0.451822,0.430182,1.804387,1.336882,-0.616191,0.611604,-0.714319,-0.098128


,method,representation,n,pearson,spearman,rmse,mae,bias,sign_accuracy,mean_pred,mean_exp
11,FoldX,structure,1234,0.319347,0.387972,2.2747,1.530312,-0.269746,0.641815,-0.3687,-0.098955


## 4. Локальный smoke/sample run

Начинайте с `sample_n = 1..3`. Если FoldX стабильно работает, увеличивайте `sample_n`. Результаты пишутся в `analysis/s669_local_run/` и повторно используются, если `force_rerun = False`.

In [ ]:
# 4. LOCAL RUN
from pipeline.s669_local_run import run_s669_local_sample, summarize_local_results

local_results = run_s669_local_sample(
    sample_n=sample_n,
    include_rosetta_score=include_rosetta_score,
    output_dir=output_dir,
    force=force_rerun,
)
display(local_results)

local_metrics = summarize_local_results(local_results)
display(local_metrics)

results_path = output_dir / 's669_local_results_long.csv'
metrics_path = output_dir / 's669_local_metrics.csv'
local_metrics.to_csv(metrics_path, index=False)
print(results_path)
print(metrics_path)


## 5. Масштабирование

Для полного локального FoldX-прогона по S669 увеличьте `sample_n` до `len(s669)` и оставьте `include_rosetta_score = False`. Rosetta score_jd2 включайте только для diagnostic score, не для финального ddG-вывода.

In [ ]:
# 5. FULL RUN SWITCH - intentionally disabled by default.
run_full_foldx = False

if run_full_foldx:
    full_results = run_s669_local_sample(
        sample_n=len(s669),
        include_rosetta_score=False,
        output_dir=output_dir,
        force=False,
    )
    full_metrics = summarize_local_results(full_results)
    display(full_metrics)
else:
    print('Full FoldX run is disabled. Set run_full_foldx = True after smoke run looks correct.')
